In [23]:
import pygadm
import pandas as pd
import geopandas as gpd
import pycountry
from pathlib import Path
import os

from bokeh.plotting import output_notebook, output_file
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.transform import factor_cmap
from bokeh.palettes import Set2

output_notebook()

Loading BokehJS ...

In [24]:
df = pd.read_parquet("../../results/country_confustion_matrix.parquet")

In [25]:
df["f1"]  = 2 * df.TP / (2 * df.TP + df.FP + df.FN)

In [26]:
import pycountry_convert as pc

def country_to_continent(country_name):
    try:
        country_code = pc.country_name_to_country_alpha2(country_name)
        continent_code = pc.country_alpha2_to_continent_code(country_code)
        continent_name = pc.convert_continent_code_to_continent_name(continent_code)
        return continent_name
    except Exception as e:
        return None  # or handle the exception/log as needed

In [27]:
df["continent"] = df["country"].apply(country_to_continent)
df = df[~df.continent.isna()]

In [28]:
hdi_df = pd.read_csv("../../data/human_development_index.csv")[["iso3", "hdi_2020"]]

In [29]:
df = pd.merge(left=df, right=hdi_df, how="left", left_on="gid", right_on="iso3")
df = df[~df.hdi_2020.isna()].rename(columns={"hdi_2020": "hdi"}).drop(columns="iso3")

In [30]:
df.head(5)

,country,gid,pixel_count,TN,FP,FN,TP,f1,continent,hdi
0,Palau,PLW,2275,2114,46,88,99,0.596386,Oceania,0.773
1,Philippines,PHL,1416080,1224155,10158,135658,46109,0.387415,Asia,0.710
2,Panama,PAN,356926,336815,5401,5747,8963,0.616565,North America,0.801
3,Oman,OMN,1543598,1470891,43239,6194,23274,0.484971,Asia,0.827
4,Nepal,NPL,782268,629907,3010,140731,8620,0.107093,Asia,0.604


In [31]:
import numpy as np


df['size'] = np.log1p(df['pixel_count'])

In [32]:
continents = df['continent'].unique().tolist()
palette = Set2[max(3, len(continents))]

source = ColumnDataSource.from_df(df)


fig = figure(
    x_axis_label='Human Development Index', 
    y_axis_label='F1',
    width=980
)
fig.scatter(
    x="hdi",
    y="f1", 
    source=source,
    alpha=0.8,
    color=factor_cmap('continent', palette=palette, factors=continents),
    legend_field='continent',
    size="size"
)

hover = HoverTool(tooltips=[
    ("Country", "@country"),
    ("F1", "@f1"),
    ("Pixel Count", "@pixel_count"),
])

fig.legend.location = "bottom_right"

fig.add_tools(hover)
show(fig)

*size of the circles is `log(total pixel count)`